# Merging files using Pandas 

In this notebook, we see a demonstration of merging files together. First, we merge all the data from the endowment files. Once we've merged all the endowment files, we'll combine columns and clean up the data as much as possible. After, we will look at all the entries from california and merge that restricted data frame with the NCES data from California. 

## Import Endowment Files
Here, we import the endowment files but eliminate the rows with information not related to our dataframe. Notice the headers start lower than usual and we cut off the files at some point (corresponding to the last sample). You can see at the end of this file all the print statements to assess the tails of the dataframes. 

In [1]:
import pandas as pd

# Load each file into a DataFrame; remove the last few rows that contain summary information
df_2018 = pd.read_excel("2018 Endowment Market Values for Media--FINAL--Revised June 11 2019.xlsx", header=5)
df_2018 = df_2018.iloc[:810]
df_2019 = pd.read_excel("2019 NTSE Endowment Market Values--FINAL.xlsx")
df_2019 = df_2019.iloc[:784]
df_2020 = pd.read_excel("2020 NTSE Endowment Market Values REVISED APRIL 22 2021.xlsx")
df_2020 = df_2020.iloc[:716]
df_2021 = pd.read_excel("2021 NTSE Endowment Market Values US and Canadian Institutions--REVISED March 1 2022.xlsx", header=9)
df_2021 = df_2021.iloc[:735]
df_2023 = pd.read_excel("2023 NCSE Endowment Market Values-FINAL.xlsx", header=9)
df_2023 = df_2023.iloc[:699]

# print(df_2018.columns)
# print(df_2019.columns)
# print(df_2020.columns)
# print(df_2021.columns)
# print(df_2023.columns)

# df_2018.tail()
# df_2019.tail()
# df_2020.tail()
# df_2021.tail()
# df_2023.tail()

## Standardizing

THe unique identifier that allows us to combine samples from all the files togehter is the IPEDS Unit ID. Unfortunately, this has a different column name in every file. So we need to standardize it. We will also do several other steps to standardize the content.

In [2]:
# Rename the unit ID columns to have the same name
df_2018.rename(columns={'Unitid1': 'UnitID'}, inplace=True)
df_2019.rename(columns={'UnitID2': 'UnitID'}, inplace=True)
df_2020.rename(columns={'UNITID2': 'UnitID'}, inplace=True)
df_2021.rename(columns={'IPEDS UNITID2': 'UnitID'}, inplace=True)
df_2023.rename(columns={'IPEDS Unit ID2': 'UnitID'}, inplace=True)

# print(df_2018.columns)
# print(df_2019.columns)
# print(df_2020.columns)
# print(df_2021.columns)
# print(df_2023.columns)

In [3]:
# Rename the Institution Type columns to have the same name
df_2021.rename(columns={'NTSE Respondent Type': 'Institution Type'}, inplace=True)
df_2023.rename(columns={'NCSE Respondent Type': 'Institution Type'}, inplace=True)

# Rename IPEDS Sector columns to have the same name
df_2018.rename(columns={'IPEDS Sector4': 'IPEDS Sector'}, inplace=True)
# no column present for 2019, 2020
df_2021.rename(columns={'IPEDS Institution Sector': 'IPEDS Sector'}, inplace=True)
df_2023.rename(columns={'IPEDS Institution Sector': 'IPEDS Sector'}, inplace=True)

# Rename HBCU columns to have the same name
df_2018.rename(columns={'IPEDS HBCU Indicator5': 'IPEDS HBCU Indicator'}, inplace=True)
df_2019.rename(columns={'IPEDS HBCU Indicator4': 'IPEDS HBCU Indicator'}, inplace=True)
df_2020.rename(columns={'Historically Black College/University (HBCU) Indicator4': 'IPEDS HBCU Indicator'}, inplace=True)
df_2021.rename(columns={'IPEDS Carnegie Classification5': 'IPEDS HBCU Indicator'}, inplace=True)
df_2023.rename(columns={'IPEDS Historically Black College/University (HBCU) Indicator4': 'IPEDS HBCU Indicator'}, inplace=True)

# Rename the Carnegie Classification columns to have the same name
df_2018.rename(columns={'IPEDS Carnegie Classification6': 'IPEDS Carnegie Classification'}, inplace=True)
df_2019.rename(columns={'IPEDS 2018 Carnegie Classification5': 'IPEDS Carnegie Classification'}, inplace=True)
df_2020.rename(columns={'2019-20 Carnegie Classification5': 'IPEDS Carnegie Classification'}, inplace=True)
df_2021.rename(columns={'IPEDS Carnegie Classification5': 'IPEDS Carnegie Classification'}, inplace=True)
df_2023.rename(columns={'IPEDS Carnegie Classification5': 'IPEDS Carnegie Classification'}, inplace=True)


In [4]:
# earlier files use numbers to indicate data. Let's change them all to strings for consistency
df_2018['IPEDS Sector'] = df_2018['IPEDS Sector'].map({0: 'State System Office/Administration Unit', 1:'4-year public college or university', 2: '4-year private non-profit college or university', 4:'2-year public college (community college)', 5:'2-year private non-profit college'})
df_2018['IPEDS HBCU Indicator'] = df_2018['IPEDS HBCU Indicator'].map({2: 'Non-HBCU', 1: 'HBCU'})
df_2018['IPEDS Carnegie Classification'] = df_2018['IPEDS Carnegie Classification'].map({0: 'State system office', 1:'Associates', 2: 'Doctoral/Research', 3:"Master's", 4: 'Baccalaureate', 5: 'Special focus'})

df_2019['IPEDS HBCU Indicator'] = df_2019['IPEDS HBCU Indicator'].map({2: 'Non-HBCU', 1: 'HBCU', 3: 'Missing/Not Available'})
df_2019['IPEDS Carnegie Classification'] = df_2019['IPEDS Carnegie Classification'].map({0: 'State system office', 1:'Associates', 2: 'Baccalaureate', 3:"Master's", 4: 'Doctoral/Research', 5: 'Special focus', 6: 'Other or Not Available'})

df_2020['IPEDS HBCU Indicator'] = df_2020['IPEDS HBCU Indicator'].map({2: 'Non-HBCU', 1: 'HBCU'})
df_2020['IPEDS Carnegie Classification'] = df_2020['IPEDS Carnegie Classification'].map({0: 'State system office', 1:'Associates', 2: 'Baccalaureate', 3:"Master's", 4: 'Doctoral/Research', 5: 'Special focus', 0: 'Not Applicable'})

## Missing Values for `UnitID`

We don't want missing `UnitID` values because we plan to merge our dataframes based on these values. If a sample has a missing Unit ID, we want to give it a new ID number that allows us to identify the same samples across all the dataframes. 

In order to achieve this, we first want to identify all the samples with the missing unit IDs. To do this, we will store all the dataframe pointers in a list (called `dfs` for "dataframes"). Then we define `indexes` as a list of lists. The entries of `indexes` are a list of sample indexes corresponding to the samples with missing `UnitID` values for the corresponding dataframe in `df`.

In [5]:
dfs = [df_2018, df_2019, df_2020, df_2021, df_2023] 
indexes = []

# indexes of samples with missing values in UnitID column
for df in dfs:
    indexes.append(df[df['UnitID'].isnull()].index)

# print(indexes)

## Assigning a Unique ID for Samples Missing the `UnitID`

This is a great application with sets! Recall sets from Computer Science 1. A **set** is a sequential datatype that stores unique elements in an unordered collection, meaning it does not maintain any specific sequence or order of elements, and duplicate entries are automatically removed.

The strategy we will use is to collect all the institution names in a set. Then assign each name with a unique integer `1` through `n` where `n` is the number of unique names. Once we have the unique values, we impute the value into the `UnitID` section. 

In [6]:
# Identify all institutions with missing IDs 
missing_institution_names = set()
for i in range(len(dfs)):
    missing_institution_names.update(dfs[i].loc[indexes[i], 'Institution Name'])

# construct a dictionary with the names in set and assign a new UnitID to each
missing_institution_names = list(missing_institution_names)
missing_institution_names_dict = {name: i for i, name in enumerate(missing_institution_names, 1)}

In [7]:
# two I looked up in CA and found the correct ID:
missing_institution_names_dict['Gateway Seminary'] = 115047
missing_institution_names_dict['The RAND Corporation'] = 121628

# check 
# for col in missing_institution_names_dict:
#     if "RAND" in col or "Gateway" in col:
#         print(col, missing_institution_names_dict[col])

In [8]:

# impute the missing values with the ids in missing_institution_names_dict
for i in range(0, len(dfs)):
    for index in indexes[i]:
        name = dfs[i].loc[index, 'Institution Name']
        dfs[i].at[index, 'UnitID'] = missing_institution_names_dict[name]

Next were some checks to make sure this was done correctly. 

In [9]:
# # checks
# for i in range(1, len(dfs)):
#     print(dfs[i][dfs[i]['UnitID'].isnull()])

# dfs[0].loc[indexes[0]]

# print(missing_institution_names_dict)

## Merging Dataframes

In the next few cells, we merge dataframes with some checks in between. Notice that we are merging using `UnitID`. We add suffixes to use in case the data we are adding has a column with the same name. So if both have a column called `Rank`, then this will be altered to be `Rank_2019` for example to avoid any conflicts. 

In [10]:
common_cols = set(dfs[0].columns)
for df in dfs[1:]:
    set_cols = set(df.columns)
    common_cols = common_cols.intersection(set_cols)

print(common_cols) # for reference later on. 

{'Institution Name', 'State', 'UnitID', 'City', 'IPEDS HBCU Indicator', 'Rank'}


In [11]:
# Perform a merge on the specified columns
merged_df = df_2023.merge(df_2021, on='UnitID' , suffixes=('_2023', '_2021'), how='outer')
merged_df = merged_df.merge(df_2020, on='UnitID', how='outer', suffixes=('', '_2020'))
merged_df = merged_df.merge(df_2019, on='UnitID', how='outer', suffixes=('', '_2019'))
merged_df = merged_df.merge(df_2018, on='UnitID', how='outer', suffixes=('', '_2018'))

# # check
# print(merged_df.head())
# print(merged_df.loc[1])

# 2018 does not have institution type 


In [12]:
merged_df.tail()

,Rank_2023,UnitID,Institution Name_2023,City_2023,State_2023,Institution Type_2023,Fall 2022 Full-time Equivalent (FTE) Enrollment3,"FY23 Total Endowment Market Value (in $1,000s)","FY22 Total Endowment Market Value (in $1,000s)",Change in Total Endowment Market Value (%)1_2023,...,City_2018,State_2018,"FY18 Endowment (in $1,000s)","FY17 Endowment (in $1,000s)",Change in Market Value (%)2,Fall 2017 Full-time Equivalent (FTE) Students3,FY18 Endowment Value per FTE Student ($),IPEDS Sector,IPEDS HBCU Indicator_2018,IPEDS Carnegie Classification_2018
947,NaN,450304.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Fort Worth,TX,84861.0,81801.0,3.740786,154.0,551045.454545,4-year private non-profit college or university,Non-HBCU,Special focus
948,476,486840.0,Kennesaw State University,Kennesaw,GA,Institutionally-related foundation (IRF),NaN,106334.537,89245.895,19.147819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
949,660,487524.0,Husson University,Bangor,ME,Private college/university endowment,2777,27313.383,23487.677,16.288141,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
950,NaN,487542.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
951,93,492263.0,The University of Tennessee System,Knoxville,TN,Public college university or system fund,55098,1599767.722,1501495.615,6.544948,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
merged_df.columns

Index(['Rank_2023', 'UnitID', 'Institution Name_2023', 'City_2023',
       'State_2023', 'Institution Type_2023',
       'Fall 2022 Full-time Equivalent (FTE) Enrollment3',
       'FY23 Total Endowment Market Value (in $1,000s)',
       'FY22 Total Endowment Market Value (in $1,000s)',
       'Change in Total Endowment Market Value (%)1_2023',
       'FY23 Endowment Value Per Full-time Equivalent (FTE) Student ($)3',
       'IPEDS Sector_2023', 'IPEDS HBCU Indicator_2023',
       'IPEDS Carnegie Classification', 'Rank_2021', 'Institution Name_2021',
       'City_2021', 'State_2021', 'Institution Type_2021',
       'Fall 2020 Full-time Equivalent (FTE) Enrollment3',
       'FY21 Total Endowment Market Value (in $1,000s)',
       'FY20 Total Endowment Market Value (in $1,000s)',
       'Change in Total Endowment Market Value (%)1_2021',
       'FY21 Endowment Value Per Full-time Equivalent Student ($)',
       'IPEDS Sector_2021',
       'IPEDS Historically Black College/University (HBCU

## Dealing with Redundant Columns

Some of the columns contain the same information. So here, we look at content related to the instution name, the city and the state. Then we reorder to columns to bring the UnitID, institution name, city, and state as the first four columns in the dataframe. 

In [14]:
def combine_columns(df, columns_to_combine, new_column_name):
    """
    Combines multiple columns in a DataFrame, keeping the first non-null value from each row,
    and drops the original columns after combining.

    Parameters:
    df (pd.DataFrame): The DataFrame with the columns to combine.
    columns_to_combine (list): A list of column names to combine.
    new_column_name (str): The name of the new combined column.

    Returns:
    pd.DataFrame: The DataFrame with combined column and original columns dropped.
    """

    # If there are no columns to combine, return the DataFrame as is
    if not columns_to_combine:
        return df

    df["temp"] = df[columns_to_combine[0]]

    # Iteratively combine other columns, prioritizing the first column's values and filling missing values from others
    for col in columns_to_combine[1:]:
        df["temp"] = df["temp"].combine_first(df[col])

    # Drop the original columns after combining
    df.drop(columns=columns_to_combine, inplace=True)

    # Rename the combined column to the specified name
    df.rename(columns={"temp": new_column_name}, inplace=True)

    return df

In [15]:
# Institution_Name columns are the same, so we can combine them
new_col_name = 'Institution Name'
institution_cols = [col for col in merged_df.columns if 'Institution Name' in col]

merged_df = combine_columns(merged_df, institution_cols, 'Institution Name')


In [16]:
# City columns are the same, so we can combine them
city_cols = [col for col in merged_df.columns if 'City' in col]
merged_df = combine_columns(merged_df, city_cols, 'City')

In [17]:
state_cols = [cols for cols in merged_df.columns if 'State' in cols]
merged_df = combine_columns(merged_df, state_cols, 'State')


In [18]:
HBCU_cols = [col for col in merged_df.columns if "HBCU" in col]
merged_df = combine_columns(merged_df, HBCU_cols, 'IPEDS HBCU Indicator')

In [19]:
type_cols = [col for col in merged_df.columns if 'Institution Type' in col]
merged_df = combine_columns(merged_df, type_cols, 'Institution Type')

In [20]:
carnegie_cols = [col for col in merged_df.columns if 'Carnegie' in col]
merged_df = combine_columns(merged_df, carnegie_cols, 'IPEDS Carnegie Classification')


In [21]:
sector_cols = [col for col in merged_df.columns if "Sector" in col]
merged_df = combine_columns(merged_df, sector_cols, 'IPEDS Sector')

In [22]:
# column name fix: Rank -> Rank_2020
merged_df.rename(columns={'Rank': 'Rank_2020'}, inplace=True)

# for col in merged_df.columns:
#     if 'Rank' in col:
#         print(col)

In [23]:
# Reorder the columns 
columns = merged_df.columns.tolist()
ordered_columns = ['UnitID', 'Institution Name', 'City', 'State', 'Institution Type', 'IPEDS Carnegie Classification', 'IPEDS HBCU Indicator', 'IPEDS Sector']
remaining_columns = [col for col in columns if col not in ordered_columns]

new_column_order = ordered_columns + remaining_columns
merged_df = merged_df[new_column_order]

print(merged_df.columns)

Index(['UnitID', 'Institution Name', 'City', 'State', 'Institution Type',
       'IPEDS Carnegie Classification', 'IPEDS HBCU Indicator', 'IPEDS Sector',
       'Rank_2023', 'Fall 2022 Full-time Equivalent (FTE) Enrollment3',
       'FY23 Total Endowment Market Value (in $1,000s)',
       'FY22 Total Endowment Market Value (in $1,000s)',
       'Change in Total Endowment Market Value (%)1_2023',
       'FY23 Endowment Value Per Full-time Equivalent (FTE) Student ($)3',
       'Rank_2021', 'Fall 2020 Full-time Equivalent (FTE) Enrollment3',
       'FY21 Total Endowment Market Value (in $1,000s)',
       'FY20 Total Endowment Market Value (in $1,000s)',
       'Change in Total Endowment Market Value (%)1_2021',
       'FY21 Endowment Value Per Full-time Equivalent Student ($)',
       'Rank_2020', 'Fall 2019 Full-time Equivalent (FTE) Enrollment3',
       'FY20 Total Endowment Market Value (in $1,000s)_2020',
       'FY19 Total Endowment Market Value (in $1,000s)',
       'Change in Tota

## Saving the Files 

The first cell saves all the merged data into a single file, which is useful if we wanted to look at several states at once. 

The second cell saves the data associated to `CA`. 

In [24]:
merged_df.to_excel("endowment_data_merged.xlsx", index=False)

In [25]:
# save a csv of merged_df where State = 'CA' 
merged_df[merged_df['State'] == 'CA'].to_csv('CAData_endowments.csv', index=False)

df_endowment_CA = merged_df[merged_df['State'] == 'CA']

# Merging the Endowment Data with the CA Data from NCES

The files from NCES contained id codes within the link. We will remove the link portion, keep the code, and refer to it as the UnitID. This will allow us to merge the data with the endowment information we constructed from above. 

In [26]:
# merge df_endowment_CA with CAData_clean.csv
df_CAData = pd.read_csv("CAData_clean.csv", header=0)

# merge
df_CA_merged = df_endowment_CA.merge(df_CAData, on='UnitID', how='outer')

## Cleaning Up Duplicates

Now we can clear up duplicate columns. 

In [27]:
state_cols = [col for col in df_CA_merged.columns if "State" in col]
df_CA_merged = combine_columns(df_CA_merged, state_cols, 'State') 

In [28]:
city_cols = [col for col in df_CA_merged.columns if "City" in col]    
institution_name_cols = [col for col in df_CA_merged.columns if "Name" in col]

df_CA_merged = combine_columns(df_CA_merged, city_cols, 'City')
df_CA_merged = combine_columns(df_CA_merged, institution_name_cols, 'Institution Name')

## Reorder columns so they are easier to work with. 

Now we reorder the columns of the merged dataframe so they seem sensible. 

In [29]:
# reorder columns: UnitID, Institution Name, City, State, Zipcode
columns = df_CA_merged.columns.tolist()
ordered_columns = ['UnitID', 'Institution Name', 'Address', 'City', 'State', 'Zipcode']
remaining_columns = [col for col in columns if col not in ordered_columns]

new_column_order = ordered_columns + remaining_columns
df_CA_merged = df_CA_merged[new_column_order]

## Save to File

Finally, we save our work to a file. 

In [30]:
# save the merged data to a csv file
df_CA_merged.to_csv('CAData_merged.csv', index=False)

Note that this method is not perfect and does warrant a visual inspection of the data! So even when you do this process by code, you can't avoid the need for a visual inspection. 

This was not expected for the assignment, which was why everyone was given smaller states. Now that you have this model, however, you can review this code and figure out how to merge files together. 